# Pa-alpha rotation curve from NIRCam WFSS, one sourceFits a disk velocity field to the R and C dispersion directions of a singlesource and reports a posterior, convergence gates and a rotation-vs-no-rotationtest. Set `SOURCE` below and run top to bottom.The two directions are never co-added: they are perpendicular projections ofthe same velocity field, and DINGO's loss is the difference between them aftereach is rectified back to the direct-image frame.**Inputs** `data/ID<id>/` must contain `config_kinematics.yaml`, `R.fits`,`C.fits`, `image.fits`, and `results/ID<id>/` a pysersic fit for theinclination. **Outputs** go to `results/ID<id>/`.

## Configuration

In [ ]:
SOURCE  = 16887        # EDR ID
DEVICE  = 'cuda'       # 'cuda' or 'cpu'
N_TEETH = 6            # staged fits run from the best N grid minima

# None = take the value from the config's `mcmc:` block.
NSTEPS = None
NBURN  = None

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

REPO = os.environ.get('DINGO_REPO', os.getcwd())
os.environ.setdefault('GRISM_CAL_DIR', os.path.join(os.path.dirname(REPO), 'grism_cal'))
sys.path.insert(0, REPO)

from dingo import fitting, plot, startpoint

DATA = os.path.join(REPO, 'data', f'ID{SOURCE}')
OUT  = os.path.join(REPO, 'results', f'ID{SOURCE}')
os.makedirs(OUT, exist_ok=True)
CONFIG = os.path.join(DATA, 'config_kinematics.yaml')
out = lambda name: os.path.join(OUT, f'ID{SOURCE}_{name}')

# config `path:` entries are relative to data/
os.chdir(os.path.join(REPO, 'data'))

for f in ('config_kinematics.yaml', 'R.fits', 'C.fits', 'image.fits'):
    p = os.path.join(DATA, f)
    print(f'{"OK " if os.path.exists(p) else "MISSING"}  {f}')

## InclinationThe grism measures `V*sin(i)`, so the amplitude is degenerate with inclination(corr ~ -0.99) unless `i` comes from imaging. It is pinned before Adam runs, sothe MAP, the frozen noise model and the posterior are all conditional on thesame geometry.

In [ ]:
sersic = np.load(out('sersic_F444W_pysersic_fixedinc.npz'), allow_pickle=True)
INC_RAD   = float(np.median(sersic['inc']))
INC_SIGMA = float(np.std(sersic['inc']))
print(f'i = {np.degrees(INC_RAD):.2f} +- {np.degrees(INC_SIGMA):.2f} deg '
      f'  (q = {np.median(sersic["q"]):.4f}, n = {np.median(sersic["n"]):.2f})')

## 1. Starting pointThe R-C objective is periodic in the velocity amplitude: a shift of onedispersion pixel re-registers R against C onto the neighbouring pixel fornearly the same residual, so the loss carries a row of near-degenerate minima.A single-start optimiser falls into whichever tooth it began in and reportsconvergence from there, and every downstream diagnostic then looks healthy.`find_start` scans `(amplitude, theta_v)` with forward evaluations only, thenruns the staged fit from each distinct minimum and keeps the lowest loss.`theta_v` is scanned because `(V, theta)` and `(-V, theta+pi)` are the samefield -- which is what makes a per-source sign table unnecessary.

In [ ]:
build = lambda path: fitting.fix_inclination(
    fitting.KinematicsFitter(config_path=path, device=DEVICE), INC_RAD)

params, loss, trials = startpoint.find_start(
    CONFIG, device=DEVICE, n_keep=N_TEETH, make_fitter=build)

amp_key = startpoint.amplitude_name(build(CONFIG).config)
print(f'{"start":>8} {"final loss":>12} {amp_key:>10}')
for t in sorted(trials, key=lambda t: t['loss']):
    print(f'{t["start"][0]:8.1f} {t["loss"]:12.6f} {t["params"][amp_key]:10.2f}')

spread = max(t['loss'] for t in trials) - loss
print(f'\nbest loss {loss:.6f}; {len(trials)} teeth spanning {spread:.6f}')

## 2. PosteriorThe fitter is rebuilt at the winning starting point, so `setup_noise_model`freezes the mask and per-pixel variance there rather than at an arbitrary tooth.

In [ ]:
fit = build(CONFIG)
fit.set_params_dict({f'velocity.{k}': v for k, v in params.items()
                     if f'velocity.{k}' in fit._mcmc_stage_cfg()})
fit.fit_all()
fit.setup_noise_model()

kw = {k: v for k, v in (('nsteps', NSTEPS), ('nburn', NBURN)) if v is not None}
fit.fit_MCMC(**kw)
print(f'mean acceptance = {np.mean(fit.mcmc_sampler.acceptance_fraction):.3f}')

## 3. GatesAcceptance outside 0.15-0.25, any `n/tau < 50`, or a split-half shift above 0.2means the widths below are not quotable.

In [ ]:
d = fit.get_mcmc_diagnostics()
acc = d['acceptance_fraction'][0]
print(f"{d['n_steps']} steps after {d['n_burn']} burn   acceptance {acc:.3f}")
print(f"fraction of -inf evaluations: {d['frac_neg_inf']:.4f}\n")
print(f'{"parameter":<22}{"tau":>8}{"n/tau":>9}{"split shift":>13}  flag')
for n in fit.mcmc_param_names:
    ntau, shift = d['n_steps_over_tau'][n], d['split_half_shift'][n]
    flag = ', '.join(f for f, bad in (('n/tau<50', ntau <= 50),
                                      ('drifting', shift > 0.2)) if bad)
    print(f'{n:<22}{d["tau"][n]:>8.1f}{ntau:>9.1f}{shift:>13.3f}  {flag}')

worst = min(d['n_steps_over_tau'].values())
gates = {'acceptance 0.15-0.25': 0.15 <= acc <= 0.25, 'worst n/tau > 50': worst > 50}
print('\n' + '\n'.join(f'{"PASS" if ok else "FAIL"}  {k}' for k, ok in gates.items()))

## 4. Does it rotate?`compare_null_model` re-optimises a `V = 0` model on the same frozen noise modeland multi-starts its offsets, because a systemic shift lowers the R-C residualjust as a velocity gradient does. A negative `dlnL` means the data prefer norotation and the amplitude is an upper limit, not a measurement.

In [ ]:
fit.set_params_to_mcmc('map')
null = fit.compare_null_model()
np.savez_compressed(out('null_model_comparison.npz'),
                    **{k: v for k, v in null.items() if not isinstance(v, dict)})

print(f"dlnL (rot - null) = {null['dlnL']:+.2f}   Wilks p = {null['p_wilks']:.2e} "
      f"(~{null['sigma_equiv']:.1f} sigma)   dBIC = {null['dBIC']:+.1f}")
ROTATES = null['dlnL'] > 0 and null['dBIC'] > 0 and null['p_wilks'] < 1.35e-3
print('DETECTION' if ROTATES else
      'UPPER LIMIT -- the data do not require rotation; do not quote V_rot or M_dyn')

## 5. Results

In [ ]:
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

chain = fit.mcmc_chain                                   # flat, post-burn
names = list(fit.mcmc_param_names)
amp = chain[:, names.index(f'velocity.{amp_key}')]

# marginalise over the imaging inclination rather than fixing it at the median
rng = np.random.default_rng(0)
inc = rng.choice(np.asarray(sersic['inc']), size=len(amp))
V_sini = amp if amp_key == 'V_sini' else amp*np.sin(INC_RAD)
V_rot  = V_sini/np.sin(np.clip(inc, 1e-6, None))

z = float(fit.config['summary']['z'])
kpc_per_arcsec = FlatLambdaCDM(H0=70, Om0=0.3).kpc_proper_per_arcmin(z).to(u.kpc/u.arcsec).value
from astropy.constants import G
R_KPC = 5.0
M5 = ((V_rot*u.km/u.s)**2 * (R_KPC*u.kpc) / G).to(u.Msun).value
logM5 = np.log10(M5)

def q(x):
    lo, mid, hi = np.percentile(x, [16, 50, 84])
    return mid, mid-lo, hi-mid

for label, arr in (('V sin(i) [km/s]', V_sini), ('V_rot [km/s]', V_rot),
                   ('log10 M_dyn(<5kpc)', logM5)):
    m, l, h = q(arr)
    print(f'{label:<22} {m:9.3f} -{l:.3f} +{h:.3f}')

np.savez_compressed(out('headline.npz'), z=z, inc_phot=INC_RAD, inc_sigma=INC_SIGMA,
                    Vsini=np.median(V_sini), V_rot=np.median(V_rot), logM5=np.median(logM5),
                    rotates=ROTATES, dlnL=null['dlnL'], dBIC=null['dBIC'],
                    start_loss=loss, acceptance=acc, worst_n_over_tau=worst)
print('\nsaved ->', out('headline.npz'))

## 6. Plots

In [ ]:
plot.plot_mcmc_chains(fit.mcmc_sampler.get_chain(), names,
                      nburn=d['n_burn'], filename=out('mcmc_chains.png'))

image_R, image_C, vz_R, vz_C = fit.get_fitting_results()
true_R, true_C = fit.get_true_images()
vel, disp_R, disp_C = fit.get_params()
vel = fitting.with_rotation_amplitude(vel, INC_RAD)

fig, axs = plt.subplots(2, 4, figsize=(18, 9))
plot.plot_kinematics_fitting_result(true_R, image_R, vz_R, image_C, 'R', axs[0], vel, disp_R)
plot.plot_kinematics_fitting_result(true_C, image_C, vz_C, image_R, 'C', axs[1], vel, disp_C)
fig.tight_layout()
fig.savefig(out('fit_summary.png'), dpi=140)